# PathVQA Training with CLIP-LLM (ViT-B-32 + Flan-T5-Base)

This notebook implements a Visual Question Answering (VQA) model for the PathVQA dataset using a CLIP-LLM architecture. It is optimized for Kaggle's dual T4 GPUs.

## Architecture:
- **Vision Encoder**: CLIP ViT-B-32 (`laion2b_s34b_b79k`)
- **LLM**: Flan-T5-Base
- **Projection**: Linear layer mapping CLIP features to T5 hidden space

## Evaluation:
- **Closed-ended**: Loss, Accuracy, F1 Score
- **Open-ended**: BLEU Score

In [1]:
!pip install -q open_clip_torch datasets evaluate rouge_score sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127

In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
import open_clip
from datasets import load_from_disk
from tqdm.auto import tqdm
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
import evaluate
from PIL import Image
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

# Configuration
CONFIG = {
    "clip_model": "ViT-B-32",
    "clip_pretrained": "laion2b_s34b_b79k",
    "llm_model": "google/flan-t5-base",
    "dataset_path": "/kaggle/input/preprocessed-path-vqa-flaviagiammarino/preprocessed_path_vqa",
    "batch_size": 32,
    "epochs": 10,
    "lr": 5e-5,
    "max_length": 128,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "num_gpus": torch.cuda.device_count()
}

print(f"Using {CONFIG['num_gpus']} GPUs")

2026-01-04 12:04:50.237882: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767528290.407112      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767528290.459031      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Using 2 GPUs


In [3]:
# Debug: Check dataset structure
dataset = load_from_disk(CONFIG['dataset_path'])
print("Dataset keys:", dataset.keys())
print("\nTrain dataset features:", dataset['train'].features)
print("\nFirst item keys:", dataset['train'][0].keys())
print("\nSample shapes:")
sample = dataset['train'][0]
for k, v in sample.items():
    if isinstance(v, list):
        print(f"  {k}: list of length {len(v)}")
    else:
        print(f"  {k}: {type(v).__name__} = {v}")

Loading dataset from disk:   0%|          | 0/24 [00:00<?, ?it/s]

Dataset keys: dict_keys(['train', 'validation', 'test'])

Train dataset features: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64')), 'class_labels': Value('int64'), 'task_type': Value('string'), 'pixel_values': List(List(List(Value('float32'))))}

First item keys: dict_keys(['input_ids', 'attention_mask', 'labels', 'class_labels', 'pixel_values'])

Sample shapes:
  input_ids: Tensor = tensor([  213,    33, 11501,  6269,  2640,    41,    32,  2165,  2640,    61,
         1069,    58,     1,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
          

In [4]:
class PathVQADataset(Dataset):
    """
    Wrapper for preprocessed PathVQA dataset.
    The dataset already has pixel_values, input_ids, attention_mask, and labels.
    We just need to convert them to tensors and handle the format.
    """
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Convert pixel_values to tensor [C, H, W]
        pixel_values = torch.tensor(item['pixel_values'], dtype=torch.float32)
        
        # Get pre-tokenized text data
        input_ids = torch.tensor(item['input_ids'], dtype=torch.long)
        attention_mask = torch.tensor(item['attention_mask'], dtype=torch.long)
        labels = torch.tensor(item['labels'], dtype=torch.long)
        
        # Get task_type if available, otherwise infer from labels
        if 'task_type' in item:
            task_type = item['task_type']
        else:
            # Infer from decoded answer - if it's yes/no, it's closed-ended
            labels_for_decode = labels.clone()
            labels_for_decode[labels_for_decode == -100] = 0  # Use 0 as placeholder
            # We'll set a default value since we can't decode without tokenizer here
            task_type = 'open'  # Default to open, will be determined during evaluation
        
        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "task_type": task_type
        }

In [5]:
class CLIPLLMModel(nn.Module):
    def __init__(self, clip_model_name, clip_pretrained, llm_model_name):
        super().__init__()
        # Load CLIP
        self.clip, _, self.clip_preprocess = open_clip.create_model_and_transforms(
            clip_model_name, pretrained=clip_pretrained
        )
        self.clip_visual = self.clip.visual
        
        # Load LLM
        self.llm = T5ForConditionalGeneration.from_pretrained(llm_model_name)
        self.tokenizer = T5Tokenizer.from_pretrained(llm_model_name)
        
        # Projection layer: CLIP ViT-B-32 output is 512, T5-base hidden is 768
        self.projector = nn.Linear(512, self.llm.config.d_model)
        
        # Freeze CLIP
        for param in self.clip_visual.parameters():
            param.requires_grad = False

    def forward(self, pixel_values, input_ids, attention_mask, labels=None):
        # Extract visual features from preprocessed pixel values
        with torch.no_grad():
            visual_features = self.clip_visual(pixel_values) # [B, 512]
        
        # Project to LLM space
        visual_embeddings = self.projector(visual_features).unsqueeze(1) # [B, 1, D]
        
        # Get text embeddings
        inputs_embeds = self.llm.get_input_embeddings()(input_ids) # [B, L, D]
        
        # Concatenate visual and text embeddings
        combined_embeds = torch.cat([visual_embeddings, inputs_embeds], dim=1)
        
        # Update attention mask for the extra visual token
        visual_mask = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
        combined_mask = torch.cat([visual_mask, attention_mask], dim=1)
        
        outputs = self.llm(
            inputs_embeds=combined_embeds,
            attention_mask=combined_mask,
            labels=labels
        )
        return outputs

    def generate(self, pixel_values, input_ids, attention_mask, max_new_tokens=32):
        visual_features = self.clip_visual(pixel_values)
        visual_embeddings = self.projector(visual_features).unsqueeze(1)
        inputs_embeds = self.llm.get_input_embeddings()(input_ids)
        combined_embeds = torch.cat([visual_embeddings, inputs_embeds], dim=1)
        
        visual_mask = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
        combined_mask = torch.cat([visual_mask, attention_mask], dim=1)
        
        return self.llm.generate(
            inputs_embeds=combined_embeds,
            attention_mask=combined_mask,
            max_new_tokens=max_new_tokens
        )

In [6]:
def train():
    # Initialize model
    model = CLIPLLMModel(CONFIG['clip_model'], CONFIG['clip_pretrained'], CONFIG['llm_model'])
    model = model.to(CONFIG['device'])
    
    if CONFIG['num_gpus'] > 1:
        model = nn.DataParallel(model)

    # Load dataset
    dataset = load_from_disk(CONFIG['dataset_path'])
    train_ds = PathVQADataset(dataset['train'])
    val_ds = PathVQADataset(dataset['validation'])
    
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    
    bleu = evaluate.load("sacrebleu")
    
    for epoch in range(CONFIG['epochs']):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")
        
        for batch in pbar:
            optimizer.zero_grad()
            
            pixel_values = batch['pixel_values'].to(CONFIG['device'])
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])
            
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            loss = outputs.loss.mean()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({"loss": loss.item()})
            
        print(f"Average Train Loss: {total_loss/len(train_loader):.4f}")
        
        # Validation
        evaluate_model(model, val_loader, bleu)

def evaluate_model(model, loader, bleu_metric):
    model.eval()
    closed_preds, closed_labels = [], []
    open_preds, open_targets = [], []
    total_loss = 0
    
    tokenizer = model.module.tokenizer if hasattr(model, 'module') else model.tokenizer
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            pixel_values = batch['pixel_values'].to(CONFIG['device'])
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])
            task_types = batch['task_type']
            
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            total_loss += outputs.loss.mean().item()
            
            # Generation for metrics
            gen_func = model.module.generate if hasattr(model, 'module') else model.generate
            generated_ids = gen_func(pixel_values, input_ids, attention_mask)
            preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            
            # Decode ground truth labels
            # Replace -100 with pad_token_id for decoding
            labels_for_decode = labels.clone()
            labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id
            targets = tokenizer.batch_decode(labels_for_decode, skip_special_tokens=True)
            
            for p, t, task in zip(preds, targets, task_types):
                p_clean = p.lower().strip()
                t_clean = t.lower().strip()
                
                # Determine if closed-ended by checking if answer is yes/no
                is_closed = t_clean in ['yes', 'no']
                
                if is_closed:
                    closed_preds.append(1 if p_clean == 'yes' else 0)
                    closed_labels.append(1 if t_clean == 'yes' else 0)
                else:
                    open_preds.append(p_clean)
                    open_targets.append(t_clean)
    
    avg_loss = total_loss / len(loader)
    print(f"Validation Loss: {avg_loss:.4f}")
    
    if closed_labels:
        acc = accuracy_score(closed_labels, closed_preds)
        f1 = f1_score(closed_labels, closed_preds)
        print(f"Closed-ended ({len(closed_labels)} questions):")
        print(f"  - Accuracy: {acc:.4f}")
        print(f"  - F1: {f1:.4f}")
    
    if open_targets:
        # Exact match accuracy
        exact_match = sum([1 for p, t in zip(open_preds, open_targets) if p == t]) / len(open_targets)
        
        # Word overlap (better for short answers)
        word_overlaps = []
        for p, t in zip(open_preds, open_targets):
            pred_words = set(p.split())
            target_words = set(t.split())
            if len(target_words) > 0:
                overlap = len(pred_words & target_words) / len(target_words)
                word_overlaps.append(overlap)
        
        # BLEU-1 (unigram only, better for short medical answers)
        from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
        bleu_scores = []
        smooth = SmoothingFunction().method1
        for p, t in zip(open_preds, open_targets):
            # Use weights=(1,0,0,0) for BLEU-1 only with smoothing
            score = sentence_bleu([t.split()], p.split(), weights=(1,0,0,0), smoothing_function=smooth)
            bleu_scores.append(score)
        
        print(f"Open-ended ({len(open_targets)} questions):")
        print(f"  - Exact Match: {exact_match:.4f}")
        print(f"  - Word Overlap: {np.mean(word_overlaps):.4f}")
        print(f"  - BLEU-1: {np.mean(bleu_scores):.4f}")

if __name__ == "__main__":
    train()

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Epoch 10/10:   0%|          | 0/615 [00:00<?, ?it/s]

/tmp/ipykernel_20/672669220.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pixel_values = torch.tensor(item['pixel_values'], dtype=torch.float32)
/tmp/ipykernel_20/672669220.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(item['input_ids'], dtype=torch.long)
/tmp/ipykernel_20/672669220.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(item['attention_mask'], dtype=torch.long)
/tmp/ipykernel_20/672669220.py:22: UserWarning: To copy construct from a tensor, it is recommend

Average Train Loss: 1.1130


Evaluating:   0%|          | 0/196 [00:00<?, ?it/s]

Validation Loss: 1.4864
Closed-ended (3125 questions):
  - Accuracy: 0.8646
  - F1: 0.8845
Open-ended (3134 questions):
  - Exact Match: 0.2125
  - Word Overlap: 0.2439
  - BLEU-1: 0.2361
